# Ejemplo de Generador

### - Instalar las dependencias timezonefinder, pymongo

In [2]:
!pip install timezonefinder

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.7/50.7 MB 2.1 MB/s eta 0:00:00m eta 0:00:010:00:01m
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 993.9/993.9 kB 2.1 MB/s eta 0:00:00m eta 0:00:010:00:01


In [4]:
!pip install pymongo

  Using cached pymongo-4.10.1-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (1.7 MB)
  Using cached dnspython-2.7.0-py3-none-any.whl (313 kB)


In [5]:
from timezonefinder import TimezoneFinder
import sys
from pymongo import MongoClient
from bson import json_util
import json

### 1. SIN YIELD

In [6]:
# Función para determinar la zona horaria
def get_iana_timezone(lat, lon, tf):
    timezone = tf.timezone_at(lat=lat, lng=lon)
    return timezone if timezone else "UTC"

In [7]:
def process_documents(documents):
    tf = TimezoneFinder()  # Inicializar TimezoneFinder una vez
    processed_docs = []  # Lista para almacenar los documentos procesados
    for idx, doc in enumerate(documents, start=1):
        # Inicializar zona horaria a "UTC" por defecto
        timezone = "UTC"
        # Obtener coordenadas desde el campo `address.coord`
        coordinates = doc.get("address", {}).get("coord")
        # Validar si las coordenadas son correctas
        if isinstance(coordinates, list) and len(coordinates) == 2:
            try:
                lon, lat = coordinates  # Formato esperado: [longitud, latitud]
                # Intentar obtener la zona horaria
                timezone = get_iana_timezone(lat, lon, tf)
            except Exception as e:
                # Si ocurre un error, mantener la zona horaria como "UTC"
                pass
        # Asignar la zona horaria al documento
        doc['timezone'] = timezone
        # Imprimir progreso en la misma línea
        sys.stdout.write(f"\rProcesando documento: {idx}")
        sys.stdout.flush()
        # Agregar el documento procesado a la lista
        processed_docs.append(doc)
    return processed_docs

In [8]:
# Cargar documentos desde el archivo JSON
input_file = 'rest.restaurants.json'
output_file = 'rest.restaurants.processed.json'

In [9]:
with open(input_file, 'r') as f:
    documents = json.load(f)

In [10]:
%%time
# Procesar los documentos
processed_documents = process_documents(documents)

# Guardar los documentos procesados en un nuevo archivo JSON
with open(output_file, 'w') as f:
    json.dump(processed_documents, f, indent=4)

print("\n¡Procesamiento completo!")

Procesando documento: 3772
¡Procesamiento completo!
CPU times: user 15.5 s, sys: 3.44 s, total: 18.9 s
Wall time: 16.3 s


### 2. CON YIELD

In [11]:
# Función para determinar la zona horaria
def get_iana_timezone_yield(lat, lon, tf):
    timezone = tf.timezone_at(lat=lat, lng=lon)
    return timezone if timezone else "UTC"

In [12]:
def process_documents_yield(documents):
    tf = TimezoneFinder()  # Inicializar TimezoneFinder una vez
    for idx, doc in enumerate(documents, start=1):
        # Inicializar zona horaria a "UTC" por defecto
        timezone = "UTC"
        # Obtener coordenadas desde el campo `address.coord`
        coordinates = doc.get("address", {}).get("coord")
        # Validar si las coordenadas son correctas
        if isinstance(coordinates, list) and len(coordinates) == 2:
            try:
                lon, lat = coordinates  # Formato esperado: [longitud, latitud]
                # Intentar obtener la zona horaria
                timezone = get_iana_timezone_yield(lat, lon, tf)
            except Exception as e:
                # Si ocurre un error, mantener la zona horaria como "UTC"
                pass
        # Asignar la zona horaria al documento
        doc['timezone'] = timezone
        # Imprimir progreso en la misma línea
        sys.stdout.write(f"\rProcesando documento: {idx}")
        sys.stdout.flush()
        yield doc  # Devolver el documento procesado

In [13]:
# Cargar documentos desde el archivo JSON
input_file = 'rest.restaurants.json'
output_file = 'rest.restaurants.yield.processed.json'

In [14]:
with open(input_file, 'r') as f:
    documents = json.load(f)

In [15]:
%%time
# Guardar los documentos procesados en un nuevo archivo JSON
with open(output_file, 'w') as f:
    f.write('[')  # Inicio del arreglo JSON
    first = True  # Bandera para manejar las comas entre objetos JSON
    for doc in process_documents_yield(documents):
        if not first:
            f.write(',\n')  # Agregar una coma antes de cada objeto JSON excepto el primero
        else:
            first = False
        # Serializar el documento en formato JSON
        json_str = json.dumps(doc)
        f.write(json_str)
    f.write(']')  # Fin del arreglo JSON

print("\n¡Procesamiento yield completo!")

Procesando documento: 3772
¡Procesamiento yield completo!
CPU times: user 13.1 s, sys: 3.2 s, total: 16.3 s
Wall time: 14.2 s


### - concluciones

| Característica             | Con `yield`                  | Sin `yield`                  |
|----------------------------|------------------------------|------------------------------|
| **Procesamiento**          | Documento por documento      | Todos los documentos a la vez |
| **Uso de memoria**         | Bajo                         | Alto                         |
| **Escritura en archivo**   | Incremental (en streaming)   | Todo al final                |
| **Complejidad del código** | Moderada (por uso de generadores) | Baja                         |
| **Flexibilidad post-procesado** | Limitada (uno por uno)      | Completa (lista completa)    |